In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import os
from torch.utils.data import DataLoader


In [27]:
class CNNHistogramNormalization(nn.Module):
    def __init__(self):
        super(CNNHistogramNormalization, self).__init__()
        self.conv1 = nn.Conv1d(1, 16, kernel_size=5, padding=2)
        self.conv2 = nn.Conv1d(16, 32, kernel_size=5, padding=2)
        self.conv3 = nn.Conv1d(32, 128, kernel_size = 5, padding =2)
        self.relu = nn.ReLU()
        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)
        self.fc1 = nn.Linear(128, 64)
        self.fc2 = nn.Linear(64, 2)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.relu(self.conv3(x))
        x = torch.nn.functional.normalize(x, p=2, dim=-1)
        x = self.global_avg_pool(x)
        x = x.view(x.shape[0], -1)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        x = self.softmax(x)
        return x

input_dir = "/home/sangheon/Desktop/Pansori_2025_ISMIR/PansoriData/Histogram"

label_map = {"GMjo": 0, "Ujo": 1}

song_data = {}

for song_folder in os.listdir(input_dir):
    song_path = os.path.join(input_dir, song_folder)
    if not os.path.isdir(song_path):
        continue

    X_list = []
    y_list = []

    for file in os.listdir(song_path):
        if file.endswith(".npy"):
            file_path = os.path.join(song_path, file)
            X_sample = np.load(file_path)

            if "GMjo" in file:
                y_label = label_map["GMjo"]
            elif "Ujo" in file:
                y_label = label_map["Ujo"]
            else:
                continue

            X_list.append(X_sample)
            y_list.append(y_label)

    if X_list:
        song_data[song_folder] = (torch.tensor(X_list, dtype=torch.float32).unsqueeze(1),
                                  torch.tensor(y_list, dtype=torch.long))

song_names = list(song_data.keys())
num_songs = len(song_names)

loo_losses = []
loo_accuracies = []

for i in range(num_songs):
    test_song = song_names[i]
    print(f"\n LOO Cross Validation - {test_song}")
    val_X, val_y = song_data[test_song]

    train_X = []
    train_y = []
    for j, song in enumerate(song_names):
        if i != j:
            train_X.append(song_data[song][0])
            train_y.append(song_data[song][1])

    train_X = torch.cat(train_X, dim=0)
    train_y = torch.cat(train_y, dim=0)

    model = CNNHistogramNormalization()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    criterion = nn.CrossEntropyLoss()

    for epoch in range(50):
        optimizer.zero_grad()
        outputs = model(train_X)
        loss = criterion(outputs, train_y)
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        val_output = model(val_X)
        #print("\n Validation Output:\n", val_output.numpy())
        val_loss = criterion(val_output, val_y)
        pred = torch.argmax(val_output, dim=1)
        accuracy = (pred == val_y).float().mean().item()
    loo_losses.append(val_loss.item())
    loo_accuracies.append(accuracy)

    print(f"Validation Loss: {val_loss.item():.4f}, Accuracy: {accuracy:.4f}")
    print(f"Prediction: {pred.numpy()}")
    print(f"Answer: {val_y.numpy()}")

print("\n  LOO-CV Result  🔹")
print(f"Average Validation Loss: {np.mean(loo_losses):.4f}")
print(f"Average Validation Accuracy: {np.mean(loo_accuracies):.4f}")


 LOO Cross Validation - 적벽가_군사설움타령_계면조
Validation Loss: 0.4527, Accuracy: 1.0000
Prediction: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
Answer: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]

 LOO Cross Validation - 박봉술_춘향가_천자뒤풀이_우조
Validation Loss: 1.1153, Accuracy: 0.0000
Prediction: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
Answer: [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]

 LOO Cross Validation - 김창환_춘향가_이별가_계면조
Validation Loss: 0.4275, Accuracy: 1.0000
Prediction: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
Answer: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]

 LOO Cross Validation - 조학진_수궁풍류_계면조
Validation Loss: 0.4730, Accuracy: 1.0000
Prediction: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
Answer: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]

 LOO Cross Validation - 박봉술_춘향가_방자설경_우조
Validation Loss: 1.0126, Accuracy: 0.0000
Prediction: [0 0 0 0 0 0 0 0 0 0]
Answer: [1 1 1 1 1 1 1 1 1 1]

 LOO Cross Validation - 춘향가_홍로의불_계면조
Validation Loss: 0.4202, Accuracy: 1.0000
Prediction: [0 0